In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, SpatialDropout1D
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

import numpy as np
import pandas as pd
import os
import seaborn as sns
import string
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report
from lime.lime_text import LimeTextExplainer

## Base Paper's dataset Preprocessing

In [2]:
import pandas as pd

# Load the dataset
paper_dataframe = pd.read_csv('/kaggle/input/humanllm/Final_dataset.csv')

# Replace '\r' and '\n' with an empty string in all columns
paper_dataframe = paper_dataframe.replace({'\r': '', '\n': ''}, regex=True)

In [3]:
paper_dataframe.head()

,uid,text,label
0,[urlsf_subset02]-[113472],Italy's President Sergio Mattarella has dissol...,1
1,[urlsf_subset01]-[60703],Brendan Gallagher's piano career was abruptly ...,1
2,[urlsf_subset04]-[262707],The concept for Wetsleeve(tm) was born out of ...,0
3,[urlsf_subset00]-[119531],Police officers in the United States had a tou...,1
4,[urlsf_subset06]-[188189],A woman has filed a lawsuit seeking more than ...,1


## firstly taking only first 30k datapoints

In [4]:
paper_dataframe =paper_dataframe[:30000]

In [5]:
label_counts = paper_dataframe['label'].value_counts()

# Print the counts
print("Count of label 0:", label_counts.get(0, 0))
print("Count of label 1:", label_counts.get(1, 0))

Count of label 0: 14891
Count of label 1: 15109


In [6]:
# Assuming df is your DataFrame
paper_dataframe.drop(columns=['uid'], inplace=True)

In [7]:
paper_dataframe.head()

,text,label
0,Italy's President Sergio Mattarella has dissol...,1
1,Brendan Gallagher's piano career was abruptly ...,1
2,The concept for Wetsleeve(tm) was born out of ...,0
3,Police officers in the United States had a tou...,1
4,A woman has filed a lawsuit seeking more than ...,1


# Loading HC-3 Dataset

In [8]:
# !pip install datasets
from datasets import load_dataset

ds = load_dataset("Hello-SimpleAI/HC3", "all")

# Concatenate inner lists into a single string for each element
concatenated = [' '.join(text_list) for text_list in ds['train']['human_answers']]

# Create a DataFrame with a single column named 'text'
df_human_hc3 = pd.DataFrame(concatenated, columns=['text'])
df_human_hc3['label'] = 0
df_human_hc3 = df_human_hc3[:1000]


concatenated_ = [' '.join(text_list) for text_list in ds['train']['chatgpt_answers']]

# Create a DataFrame with a single column named 'text'
df_ai_hc3 = pd.DataFrame(concatenated_, columns=['text'])
df_ai_hc3['label'] = 1
df_ai_hc3 = df_ai_hc3[:1000]

#concatenate both the dataset
df_HC3 = pd.concat([df_ai_hc3, df_human_hc3])
#shuffle the dataset
df_HC3 = df_HC3.sample(frac=1).reset_index(drop=True)
print(df_HC3.head())

0000.parquet:   0%|          | 0.00/39.3M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/24322 [00:00<?, ? examples/s]

                                                text  label
0  For compounding interest : If you have an acco...      0
1  Mass is a measure of the amount of matter in a...      1
2  Gold is : * rare , but not too rare * distinct...      0
3  Because it 's more expensive to mass - produce...      0
4  URL_0 is a placeholder that represents a speci...      1


## Our dataset preprocessing

In [9]:
# Define a function to preprocess each text
def preprocess_text(text):
    # Find the index of the first colon
    colon_index = text.find(':')
    # If a colon exists and is within the first 20 words, trim the text
    if colon_index != -1:
        # Get the words up to the colon's position
        words_before_colon = text[:colon_index].split()
        # Check if the colon is within the first 20 words
        if len(words_before_colon) <= 20:
            # Return the text after the colon, trimmed of leading/trailing spaces
            return text[colon_index + 1:].strip()
    # If no colon or colon is outside the first 20 words, return the original text
    return text



In [10]:
ai_df = pd.read_csv("/kaggle/input/3500-datapoints/updated_dataset.csv", encoding='ISO-8859-1')
ai_df = ai_df.replace({'\r': '', '\n': ''}, regex=True)
filtered_df = ai_df[ai_df['ai_generated_text'].notna()]

human_df = pd.DataFrame(filtered_df['text'])
ai_df = pd.DataFrame(filtered_df['ai_generated_text'])
ai_df['ai_generated_text'] = ai_df['ai_generated_text'].apply(preprocess_text)
ai_df.rename(columns = {'ai_generated_text':'text'}, inplace = True)
human_df['label'] = 0
ai_df['label'] = 1

#concatenate the dataset
my_df = pd.concat([human_df, ai_df])
#shuffle the dataset
my_df = my_df.sample(frac=1).reset_index(drop=True)

In [11]:
my_df.head()

,text,label
0,Photos: Photos: Emotions run high in Ferguson ...,0
1,The original Stompa designs I saw were a bit t...,1
2,Sensitivity and heterogeneity analysis In this...,0
3,It's been a long time since the Star Wars saga...,1
4,"The fact is that, unfair as it may seem, Afric...",0


In [12]:
my_df.shape

(7514, 2)

## Paper's model code

# 1. Distilbert generated 

In [13]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, f1_score, confusion_matrix
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
from transformers import DistilBertModel, DistilBertTokenizer

# Assuming 'df' is your dataset with columns ['text', 'label']
# Split the dataset into training and testing sets
print("Splitting the dataset into training and testing sets...")
train_df, test_df = train_test_split(paper_dataframe, test_size=0.2, random_state=42)
print(f"Training set size: {len(train_df)}, Testing set size: {len(test_df)}")

class TextDataset(Dataset):
    def __init__(self, dataframe):
        self.dataframe = dataframe

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        # Extract the text and label from the dataframe
        text = self.dataframe.iloc[idx, 0]  # assuming 'text' is the first column
        label = int(self.dataframe.iloc[idx, 1])  # convert label to integer
        return text, label

# Create datasets for training and testing
print("Creating datasets...")
train_dataset = TextDataset(train_df)
test_dataset = TextDataset(test_df)

# Create data loaders for batching
print("Creating data loaders...")
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

class Sentinel(nn.Module):
    def __init__(self, distilbert: DistilBertModel, tokenizer: DistilBertTokenizer) -> None:
        super().__init__()
        self.distilbert = distilbert
        # Freeze the parameters of the DistilBERT model
        for param in self.distilbert.parameters():
            param.requires_grad = False
        self.tokenizer = tokenizer

        # Define a fully connected layer for classification
        self.fc = nn.Sequential(
            nn.Linear(768, 768),
            nn.GELU(), nn.Dropout(0.25),
            nn.Linear(768, 2)
        )

    def forward(self, textBatch) -> torch.Tensor:
        # Tokenize and encode the text batch
        encodedText = self.tokenizer(
            textBatch, max_length=512, truncation=True,
            padding="max_length", return_tensors="pt"
        ).to("cuda")

        # Get the last hidden states from DistilBERT
        lastHiddenStates = self.distilbert(**encodedText).last_hidden_state
        # Pass through the fully connected layer to get logits
        logits = self.fc(lastHiddenStates[:, 0, :])
        return logits

# Load the pretrained DistilBERT model and tokenizer
print("Loading DistilBERT model and tokenizer...")
distil_model = DistilBertModel.from_pretrained('distilbert-base-uncased')
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

# Initialize the Sentinel model
print("Initializing the Sentinel model...")
distil_model = Sentinel(distil_model, tokenizer).to("cuda")

# Define the loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(distil_model.parameters(), lr=1e-4)

# Set the number of epochs for training
num_epochs = 10
distil_model.train()

print("Starting training...")
for epoch in range(num_epochs):
    running_loss = 0.0
    correct_predictions = 0
    total_predictions = 0

    for texts, labels in train_loader:
        # Convert texts to a list and move labels to GPU
        texts = list(texts)
        labels = labels.to("cuda")

        # Zero the gradients
        optimizer.zero_grad()
        # Forward pass
        outputs = distil_model(texts)
        # Compute the loss
        loss = criterion(outputs, labels)
        # Backward pass and optimization
        loss.backward()
        optimizer.step()

        # Accumulate the loss
        running_loss += loss.item()
        # Calculate the number of correct predictions
        _, predicted = torch.max(outputs, 1)
        correct_predictions += (predicted == labels).sum().item()
        total_predictions += labels.size(0)

    # Calculate average loss and accuracy for the epoch
    epoch_loss = running_loss / len(train_loader)
    epoch_accuracy = correct_predictions / total_predictions
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {epoch_loss:.4f}, Accuracy: {epoch_accuracy:.4f}")

print("Training complete")

# Save the trained model to a file
print("Saving the model...")
torch.save(distil_model.state_dict(), 'distilbert_sentinel.pth')
print("Model saved as 'distilbert_sentinel.pth'")

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

Splitting the dataset into training and testing sets...
Training set size: 24000, Testing set size: 6000
Creating datasets...
Creating data loaders...
Loading DistilBERT model and tokenizer...


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Initializing the Sentinel model...
Starting training...
Epoch 1/10, Loss: 0.3453, Accuracy: 0.8614
Epoch 2/10, Loss: 0.2432, Accuracy: 0.9043
Epoch 3/10, Loss: 0.2308, Accuracy: 0.9079
Epoch 4/10, Loss: 0.2248, Accuracy: 0.9100
Epoch 5/10, Loss: 0.2191, Accuracy: 0.9137
Epoch 6/10, Loss: 0.2171, Accuracy: 0.9140
Epoch 7/10, Loss: 0.2147, Accuracy: 0.9136
Epoch 8/10, Loss: 0.2079, Accuracy: 0.9178
Epoch 9/10, Loss: 0.2080, Accuracy: 0.9179
Epoch 10/10, Loss: 0.2081, Accuracy: 0.9188
Training complete
Saving the model...
Model saved as 'distilbert_sentinel.pth'


In [14]:
# Step 5: Evaluate on the test set
distil_model.eval()
all_labels = []
all_predictions = []

with torch.no_grad():
    for texts, labels in test_loader:
        texts = list(texts)
        labels = labels.to("cuda")

        outputs = distil_model(texts)
        _, predicted = torch.max(outputs, 1)

        all_labels.extend(labels.cpu().numpy())
        all_predictions.extend(predicted.cpu().numpy())

# Classification report
print("Classification Report:")
print(classification_report(all_labels, all_predictions))

# Calculate F1 Score, FNR, and FPR
f1 = f1_score(all_labels, all_predictions, average='weighted')
cm = confusion_matrix(all_labels, all_predictions)
tn, fp, fn, tp = cm.ravel()
fnr = fn / (fn + tp)
fpr = fp / (fp + tn)

print(f"F1 Score: {f1:.4f}")
print(f"False Negative Rate (FNR): {fnr:.4f}")
print(f"False Positive Rate (FPR): {fpr:.4f}")

Classification Report:
              precision    recall  f1-score   support

           0       0.96      0.86      0.91      2909
           1       0.88      0.97      0.92      3091

    accuracy                           0.91      6000
   macro avg       0.92      0.91      0.91      6000
weighted avg       0.92      0.91      0.91      6000

F1 Score: 0.9143
False Negative Rate (FNR): 0.0320
False Positive Rate (FPR): 0.1420


# DistilBert model trained on base paper's dataset and testing on our dataset(5k datapoints)

In [15]:
# No need of loading dataset again and again.Use defined our dataframe 'my_df'

# ai_df = pd.read_csv("/kaggle/input/3500-datapoints/updated_dataset.csv", encoding='ISO-8859-1')
# ai_df = ai_df.replace({'\r': '', '\n': ''}, regex=True)
# filtered_df = ai_df[ai_df['ai_generated_text'].notna()]
# human_df = pd.DataFrame(filtered_df['text'])
# ai_df = pd.DataFrame(filtered_df['ai_generated_text'])
# ai_df['ai_generated_text'] = ai_df['ai_generated_text'].apply(preprocess_text)
# ai_df.rename(columns = {'ai_generated_text':'text'}, inplace = True)
# human_df['label'] = 0
# ai_df['label'] = 1
# my_df = pd.concat([human_df[:1000], ai_df[:1000]], ignore_index=True)

my_df= my_df[:5000]



class TextDataset(Dataset):
    def __init__(self, dataframe):
        self.dataframe = dataframe

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        text = self.dataframe.iloc[idx, 0]  # assuming 'text' is the first column
        label = int(self.dataframe.iloc[idx, 1])  # convert label to integer
        return text, label


# ***** we are using our complete dataset for testing????  **************

# Data Loaders
# train_dataset = TextDataset(train_df)
test_dataset = TextDataset(my_df)

# train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)  # Single datapoint per batch
test_loader = DataLoader(test_dataset, batch_size=2, shuffle=False)

In [16]:
import torch
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report

# Load the saved model weights
# model.load_state_dict(torch.load('/kaggle/input/2_epoch_mix_model/other/default/1/our_ensemble_model.pth'))
distil_model.eval()
# DataLoader for the test dataset

# Evaluate the model on the new dataset
print("Evaluating the model on the our dataset...")
all_labels = []
all_predictions = []

with torch.no_grad():
    for texts, labels in test_loader:
        texts = list(texts)
        labels = labels.to("cuda")

        outputs = distil_model(texts)
        _, predicted = torch.max(outputs, 1)

        all_labels.extend(labels.cpu().numpy())
        all_predictions.extend(predicted.cpu().numpy())

# Classification report
print("Classification Report:")
print(classification_report(all_labels, all_predictions))

# Calculate F1 Score, FNR, and FPR
f1 = f1_score(all_labels, all_predictions, average='weighted')
cm = confusion_matrix(all_labels, all_predictions)
tn, fp, fn, tp = cm.ravel()
fnr = fn / (fn + tp)
fpr = fp / (fp + tn)

print(f"F1 Score: {f1:.4f}")
print(f"False Negative Rate (FNR): {fnr:.4f}")
print(f"False Positive Rate (FPR): {fpr:.4f}")


Evaluating the model on the our dataset...
Classification Report:
              precision    recall  f1-score   support

           0       0.88      0.91      0.89      2501
           1       0.90      0.88      0.89      2499

    accuracy                           0.89      5000
   macro avg       0.89      0.89      0.89      5000
weighted avg       0.89      0.89      0.89      5000

F1 Score: 0.8930
False Negative Rate (FNR): 0.1204
False Positive Rate (FPR): 0.0936


# distilbert model trained of base papers's dataset and testing on HC3 dataset(2k datapoints).

In [17]:

class TextDataset(Dataset):
    def __init__(self, dataframe):
        self.dataframe = dataframe

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        text = self.dataframe.iloc[idx, 0]  # assuming 'text' is the first column
        label = int(self.dataframe.iloc[idx, 1])  # convert label to integer
        return text, label

# Data Loaders
# train_dataset = TextDataset(train_df)
# ****** we are using total 2000 data points in testing dataset *********
test_dataset = TextDataset(df_HC3)

# train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)  # Single datapoint per batch
test_loader = DataLoader(test_dataset, batch_size=2, shuffle=False)

In [18]:
import torch
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report

# Load the saved model weights
# ensemble_model.load_state_dict(torch.load('/kaggle/input/2_epoch_mix_model/other/default/1/our_ensemble_model.pth'))
distil_model.eval()

# DataLoader for the test dataset
# test_loader = DataLoader(df_HC3, batch_size=2, shuffle=False)
# Evaluate the model on the new dataset


print("Evaluating the model on the HC3 dataset...")
all_labels = []
all_predictions = []

with torch.no_grad():
    for texts, labels in test_loader:
        texts = list(texts)
        labels = labels.to("cuda")

        outputs = distil_model(texts)
        _, predicted = torch.max(outputs, 1)

        all_labels.extend(labels.cpu().numpy())
        all_predictions.extend(predicted.cpu().numpy())

# Classification report
print("Classification Report:")
print(classification_report(all_labels, all_predictions))

# Calculate F1 Score, FNR, and FPR
f1 = f1_score(all_labels, all_predictions, average='weighted')
cm = confusion_matrix(all_labels, all_predictions)
tn, fp, fn, tp = cm.ravel()
fnr = fn / (fn + tp)
fpr = fp / (fp + tn)

print(f"F1 Score: {f1:.4f}")
print(f"False Negative Rate (FNR): {fnr:.4f}")
print(f"False Positive Rate (FPR): {fpr:.4f}")


Evaluating the model on the HC3 dataset...
Classification Report:
              precision    recall  f1-score   support

           0       0.97      0.92      0.94      1000
           1       0.92      0.97      0.94      1000

    accuracy                           0.94      2000
   macro avg       0.94      0.94      0.94      2000
weighted avg       0.94      0.94      0.94      2000

F1 Score: 0.9430
False Negative Rate (FNR): 0.0290
False Positive Rate (FPR): 0.0850


# 2. Roberta

In [20]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, f1_score, confusion_matrix
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
from transformers import RobertaModel, RobertaTokenizer

# Assuming 'df' is your dataset with columns ['text', 'label']
# Split the dataset into training and testing sets
print("Splitting the dataset into training and testing sets...")
train_df, test_df = train_test_split(paper_dataframe, test_size=0.2, random_state=42)
print(f"Training set size: {len(train_df)}, Testing set size: {len(test_df)}")

class TextDataset(Dataset):
    def __init__(self, dataframe):
        self.dataframe = dataframe

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        # Extract the text and label from the dataframe
        text = self.dataframe.iloc[idx, 0]  # assuming 'text' is the first column
        label = int(self.dataframe.iloc[idx, 1])  # convert label to integer
        return text, label

# Create datasets for training and testing
print("Creating datasets...")
train_dataset = TextDataset(train_df)
test_dataset = TextDataset(test_df)

# Create data loaders for batching
print("Creating data loaders...")
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

class Sentinel(nn.Module):
    def __init__(self, roberta: RobertaModel, tokenizer: RobertaTokenizer) -> None:
        super().__init__()
        self.roberta = roberta
        # Freeze the parameters of the Roberta model
        for param in self.roberta.parameters():
            param.requires_grad = False
        self.tokenizer = tokenizer

        # Define a fully connected layer for classification
        self.fc = nn.Sequential(
            nn.Linear(768, 768),
            nn.GELU(), nn.Dropout(0.25),
            nn.Linear(768, 2)
        )

    def forward(self, textBatch) -> torch.Tensor:
        # Tokenize and encode the text batch
        encodedText = self.tokenizer(
            textBatch, max_length=512, truncation=True,
            padding="max_length", return_tensors="pt"
        ).to("cuda")

        # Get the last hidden states from Roberta
        lastHiddenStates = self.roberta(**encodedText).last_hidden_state
        # Pass through the fully connected layer to get logits
        logits = self.fc(lastHiddenStates[:, 0, :])
        return logits

# Load the pretrained Roberta model and tokenizer
print("Loading Roberta model and tokenizer...")
roberta_model = RobertaModel.from_pretrained('roberta-base')
tokenizer = RobertaTokenizer.from_pretrained('roberta-base')

# Initialize the Sentinel model
print("Initializing the Sentinel model...")
roberta_model = Sentinel(roberta_model, tokenizer).to("cuda")

# Define the loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(roberta_model.parameters(), lr=1e-4)

# Set the number of epochs for training
num_epochs = 3
roberta_model.train()

print("Starting training...")
for epoch in range(num_epochs):
    running_loss = 0.0
    correct_predictions = 0
    total_predictions = 0

    for texts, labels in train_loader:
        # Convert texts to a list and move labels to GPU
        texts = list(texts)
        labels = labels.to("cuda")

        # Zero the gradients
        optimizer.zero_grad()
        # Forward pass
        outputs = roberta_model(texts)
        # Compute the loss
        loss = criterion(outputs, labels)
        # Backward pass and optimization
        loss.backward()
        optimizer.step()

        # Accumulate the loss
        running_loss += loss.item()
        # Calculate the number of correct predictions
        _, predicted = torch.max(outputs, 1)
        correct_predictions += (predicted == labels).sum().item()
        total_predictions += labels.size(0)

    # Calculate average loss and accuracy for the epoch
    epoch_loss = running_loss / len(train_loader)
    epoch_accuracy = correct_predictions / total_predictions
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {epoch_loss:.4f}, Accuracy: {epoch_accuracy:.4f}")

print("Training complete")

# Save the trained model to a file
print("Saving the model...")
torch.save(roberta_model.state_dict(), 'roberta_sentinel.pth')
print("Model saved as 'roberta_sentinel.pth'")


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Splitting the dataset into training and testing sets...
Training set size: 24000, Testing set size: 6000
Creating datasets...
Creating data loaders...
Loading Roberta model and tokenizer...
Initializing the Sentinel model...
Starting training...
Epoch 1/3, Loss: 0.5156, Accuracy: 0.8241
Epoch 2/3, Loss: 0.3138, Accuracy: 0.8967
Epoch 3/3, Loss: 0.2650, Accuracy: 0.9058
Training complete
Saving the model...
Model saved as 'roberta_sentinel.pth'


In [21]:
# Step 5: Evaluate on the test set
roberta_model.eval()
all_labels = []
all_predictions = []

with torch.no_grad():
    for texts, labels in test_loader:
        texts = list(texts)
        labels = labels.to("cuda")

        outputs = roberta_model(texts)
        _, predicted = torch.max(outputs, 1)

        all_labels.extend(labels.cpu().numpy())
        all_predictions.extend(predicted.cpu().numpy())

# Classification report
print("Classification Report:")
print(classification_report(all_labels, all_predictions))

# Calculate F1 Score, FNR, and FPR
f1 = f1_score(all_labels, all_predictions, average='weighted')
cm = confusion_matrix(all_labels, all_predictions)
tn, fp, fn, tp = cm.ravel()
fnr = fn / (fn + tp)
fpr = fp / (fp + tn)

print(f"F1 Score: {f1:.4f}")
print(f"False Negative Rate (FNR): {fnr:.4f}")
print(f"False Positive Rate (FPR): {fpr:.4f}")

Classification Report:
              precision    recall  f1-score   support

           0       0.97      0.85      0.91      2909
           1       0.87      0.98      0.92      3091

    accuracy                           0.91      6000
   macro avg       0.92      0.91      0.91      6000
weighted avg       0.92      0.91      0.91      6000

F1 Score: 0.9138
False Negative Rate (FNR): 0.0210
False Positive Rate (FPR): 0.1543


# Roberta model on base paper's dataset and testing on our dataset(5k datapoints)

In [22]:
# ai_df = pd.read_csv("/kaggle/input/3500-datapoints/updated_dataset.csv", encoding='ISO-8859-1')
# ai_df = ai_df.replace({'\r': '', '\n': ''}, regex=True)
# filtered_df = ai_df[ai_df['ai_generated_text'].notna()]
# human_df = pd.DataFrame(filtered_df['text'])
# ai_df = pd.DataFrame(filtered_df['ai_generated_text'])
# ai_df['ai_generated_text'] = ai_df['ai_generated_text'].apply(preprocess_text)
# ai_df.rename(columns = {'ai_generated_text':'text'}, inplace = True)
# human_df['label'] = 0
# ai_df['label'] = 1
# my_df = pd.concat([human_df[:1000], ai_df[:1000]], ignore_index=True)


my_df= my_df[:5000]



class TextDataset(Dataset):
    def __init__(self, dataframe):
        self.dataframe = dataframe

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        text = self.dataframe.iloc[idx, 0]  # assuming 'text' is the first column
        label = int(self.dataframe.iloc[idx, 1])  # convert label to integer
        return text, label

# Data Loaders
# train_dataset = TextDataset(train_df)
test_dataset = TextDataset(my_df)

# train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)  # Single datapoint per batch
test_loader = DataLoader(test_dataset, batch_size=2, shuffle=False)

In [23]:
import torch
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report

# Load the saved model weights
# ensemble_model.load_state_dict(torch.load('/kaggle/working/roberta_sentinel.pth'))

roberta_model.eval()

# DataLoader for the test dataset
# test_loader = DataLoader(df_HC3, batch_size=2, shuffle=False)

# Function to test the model on the test data
# Evaluate the model on the new dataset
print("Evaluating the Roberta model on the our dataset...")
all_labels = []
all_predictions = []

with torch.no_grad():
    for texts, labels in test_loader:
        texts = list(texts)
        labels = labels.to("cuda")

        outputs = roberta_model(texts)
        _, predicted = torch.max(outputs, 1)

        all_labels.extend(labels.cpu().numpy())
        all_predictions.extend(predicted.cpu().numpy())

# Classification report
print("Classification Report:")
print(classification_report(all_labels, all_predictions))

# Calculate F1 Score, FNR, and FPR
f1 = f1_score(all_labels, all_predictions, average='weighted')
cm = confusion_matrix(all_labels, all_predictions)
tn, fp, fn, tp = cm.ravel()
fnr = fn / (fn + tp)
fpr = fp / (fp + tn)

print(f"F1 Score: {f1:.4f}")
print(f"False Negative Rate (FNR): {fnr:.4f}")
print(f"False Positive Rate (FPR): {fpr:.4f}")



Evaluating the Roberta model on the our dataset...
Classification Report:
              precision    recall  f1-score   support

           0       0.80      0.40      0.54      2501
           1       0.60      0.90      0.72      2499

    accuracy                           0.65      5000
   macro avg       0.70      0.65      0.63      5000
weighted avg       0.70      0.65      0.63      5000

F1 Score: 0.6276
False Negative Rate (FNR): 0.1008
False Positive Rate (FPR): 0.5978


# Roberta model trained on base dataset and testing on HC3 dataset(2k datapoints).

In [24]:
# # !pip install datasets
# from datasets import load_dataset

# ds = load_dataset("Hello-SimpleAI/HC3", "all")

# # Concatenate inner lists into a single string for each element
# concatenated = [' '.join(text_list) for text_list in ds['train']['human_answers']]

# # Create a DataFrame with a single column named 'text'
# df_human_hc3 = pd.DataFrame(concatenated, columns=['text'])
# df_human_hc3['label'] = 0
# df_human_hc3 = df_human_hc3[:1000]


# concatenated_ = [' '.join(text_list) for text_list in ds['train']['chatgpt_answers']]

# # Create a DataFrame with a single column named 'text'
# df_ai_hc3 = pd.DataFrame(concatenated_, columns=['text'])
# df_ai_hc3['label'] = 1
# df_ai_hc3 = df_ai_hc3[:1000]



# df_HC3 = pd.concat([df_ai_hc3, df_human_hc3], ignore_index=True)
# df_HC3 = df_HC3[:100]

class TextDataset(Dataset):
    def __init__(self, dataframe):
        self.dataframe = dataframe

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        text = self.dataframe.iloc[idx, 0]  # assuming 'text' is the first column
        label = int(self.dataframe.iloc[idx, 1])  # convert label to integer
        return text, label

# Data Loaders
# train_dataset = TextDataset(train_df)
test_dataset = TextDataset(df_HC3)

# train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)  # Single datapoint per batch
test_loader = DataLoader(test_dataset, batch_size=2, shuffle=False)

In [25]:
import torch
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report

# Load the saved model weights
# ensemble_model.load_state_dict(torch.load('/kaggle/input/2_epoch_mix_model/other/default/1/our_ensemble_model.pth'))

roberta_model.eval()

# DataLoader for the test dataset
# test_loader = DataLoader(df_HC3, batch_size=2, shuffle=False)

# Evaluate the model on the new dataset
print("Evaluating the Roberta model on the HC3 dataset...")
all_labels = []
all_predictions = []

with torch.no_grad():
    for texts, labels in test_loader:
        texts = list(texts)
        labels = labels.to("cuda")

        outputs = roberta_model(texts)
        _, predicted = torch.max(outputs, 1)

        all_labels.extend(labels.cpu().numpy())
        all_predictions.extend(predicted.cpu().numpy())

# Classification report
print("Classification Report:")
print(classification_report(all_labels, all_predictions))

# Calculate F1 Score, FNR, and FPR
f1 = f1_score(all_labels, all_predictions, average='weighted')
cm = confusion_matrix(all_labels, all_predictions)
tn, fp, fn, tp = cm.ravel()
fnr = fn / (fn + tp)
fpr = fp / (fp + tn)

print(f"F1 Score: {f1:.4f}")
print(f"False Negative Rate (FNR): {fnr:.4f}")
print(f"False Positive Rate (FPR): {fpr:.4f}")


Evaluating the Roberta model on the HC3 dataset...
Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1000
           1       1.00      1.00      1.00      1000

    accuracy                           1.00      2000
   macro avg       1.00      1.00      1.00      2000
weighted avg       1.00      1.00      1.00      2000

F1 Score: 0.9995
False Negative Rate (FNR): 0.0010
False Positive Rate (FPR): 0.0000


# 3. T5

In [26]:
import torch
import torch.nn as nn
from transformers import AutoTokenizer, T5ForConditionalGeneration
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
import pandas as pd
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm


# Split into training and testing datasets
train_df, test_df = train_test_split(paper_dataframe, test_size=0.2, random_state=42)

class TextDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len):
        self.data = dataframe
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        text = self.data.iloc[index]['text']
        label = self.data.iloc[index]['label']
        return text, label

    def collate_fn(self, batch):
        texts, labels = zip(*batch)
        encoded_texts = self.tokenizer.batch_encode_plus(
            texts, max_length=self.max_len, truncation=True, padding="max_length", return_tensors="pt"
        )
        labels = torch.tensor(labels)
        return encoded_texts.input_ids, encoded_texts.attention_mask, labels

class BinaryClassifier(nn.Module):
    def __init__(self, model_name="t5-small"):
        super().__init__()
        self.t5 = T5ForConditionalGeneration.from_pretrained(model_name)
        self.classifier = nn.Linear(self.t5.config.d_model, 2)

    def forward(self, input_ids, attention_mask):
        outputs = self.t5.encoder(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs.last_hidden_state.mean(dim=1)
        logits = self.classifier(pooled_output)
        return logits

# Model, Optimizer, and Loss Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = BinaryClassifier().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
criterion = nn.CrossEntropyLoss()

# DataLoader Setup
tokenizer = AutoTokenizer.from_pretrained("t5-small")
train_dataset = TextDataset(train_df, tokenizer, max_len=512)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, collate_fn=train_dataset.collate_fn)
test_dataset = TextDataset(test_df, tokenizer, max_len=512)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, collate_fn=test_dataset.collate_fn)

def train(model, dataloader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    all_preds = []
    all_labels = []

    progress_bar = tqdm(dataloader, desc="Training")
    for input_ids, attention_mask, labels in progress_bar:
        input_ids, attention_mask, labels = input_ids.to(device), attention_mask.to(device), labels.to(device)

        optimizer.zero_grad()
        logits = model(input_ids, attention_mask)
        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        preds = torch.argmax(logits, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

        accuracy = accuracy_score(labels.cpu().numpy(), preds.cpu().numpy())
        progress_bar.set_postfix({'loss': f'{loss.item():.4f}', 'acc': f'{accuracy:.4f}'})

    epoch_loss = total_loss / len(dataloader)
    epoch_accuracy = accuracy_score(all_labels, all_preds)
    epoch_f1 = f1_score(all_labels, all_preds, average='weighted')

    return epoch_loss, epoch_accuracy, epoch_f1

def evaluate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for input_ids, attention_mask, labels in tqdm(dataloader, desc="Evaluating"):
            input_ids, attention_mask, labels = input_ids.to(device), attention_mask.to(device), labels.to(device)

            logits = model(input_ids, attention_mask)
            loss = criterion(logits, labels)

            total_loss += loss.item()

            preds = torch.argmax(logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    epoch_loss = total_loss / len(dataloader)
    epoch_accuracy = accuracy_score(all_labels, all_preds)
    epoch_f1 = f1_score(all_labels, all_preds, average='weighted')

    return epoch_loss, epoch_accuracy, epoch_f1

# Training and Evaluation
epochs = 3
for epoch in range(epochs):
    print(f"Epoch {epoch+1}/{epochs}")
    train_loss, train_accuracy, train_f1 = train(model, train_loader, optimizer, criterion, device)
    test_loss, test_accuracy, test_f1 = evaluate(model, test_loader, criterion, device)

    print(f"Train - Loss: {train_loss:.4f}, Accuracy: {train_accuracy:.4f}, F1: {train_f1:.4f}")
    print(f"Test  - Loss: {test_loss:.4f}, Accuracy: {test_accuracy:.4f}, F1: {test_f1:.4f}")
    print("-" * 50)

# Save the trained model
print("Saving the model...")
torch.save(model.state_dict(), 't5_sentinel.pth')
print("Model saved as 't5_sentinel.pth'")

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

...

Train - Loss: 0.2719, Accuracy: 0.9017, F1: 0.9016
Test  - Loss: 0.1107, Accuracy: 0.9612, F1: 0.9612
--------------------------------------------------
Epoch 2/3


Evaluating: 100%|██████████| 375/375 [00:59<00:00,  6.31it/s]


Train - Loss: 0.0959, Accuracy: 0.9671, F1: 0.9671
Test  - Loss: 0.0835, Accuracy: 0.9720, F1: 0.9720
--------------------------------------------------
Epoch 3/3


Evaluating: 100%|██████████| 375/375 [00:59<00:00,  6.35it/s]


Train - Loss: 0.0716, Accuracy: 0.9758, F1: 0.9758
Test  - Loss: 0.1017, Accuracy: 0.9675, F1: 0.9675
--------------------------------------------------
Saving the model...
Model saved as 't5_sentinel.pth'


In [27]:
# import pandas as pd
# from sklearn.model_selection import train_test_split
# from sklearn.metrics import classification_report, accuracy_score, f1_score, confusion_matrix
# import torch
# from torch.utils.data import Dataset, DataLoader
# import torch.nn as nn
# import torch.optim as optim
# from transformers import T5ForConditionalGeneration, T5Tokenizer

# # Assuming 'df' is your dataset with columns ['text', 'label']
# # Split the dataset into training and testing sets
# print("Splitting the dataset into training and testing sets...")
# train_df, test_df = train_test_split(paper_dataframe, test_size=0.2, random_state=42)
# print(f"Training set size: {len(train_df)}, Testing set size: {len(test_df)}")

# class TextDataset(Dataset):
#     def __init__(self, dataframe):
#         self.dataframe = dataframe

#     def __len__(self):
#         return len(self.dataframe)

#     def __getitem__(self, idx):
#         # Extract the text and label from the dataframe
#         text = self.dataframe.iloc[idx, 0]  # assuming 'text' is the first column
#         label = str(self.dataframe.iloc[idx, 1])  # convert label to string for T5
#         return text, label

# # Create datasets for training and testing
# print("Creating datasets...")
# train_dataset = TextDataset(train_df)
# test_dataset = TextDataset(test_df)

# # Create data loaders for batching
# print("Creating data loaders...")
# train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
# test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

# class T5Sentinel(nn.Module):
#     def __init__(self, t5_model: T5ForConditionalGeneration, tokenizer: T5Tokenizer) -> None:
#         super().__init__()
#         self.t5 = t5_model
#         self.tokenizer = tokenizer

#     def forward(self, text_batch, labels=None):
#         # Tokenize and encode the text batch
#         encoded_input = self.tokenizer(
#             text_batch, max_length=512, truncation=True,
#             padding="max_length", return_tensors="pt"
#         ).to(self.t5.device)

#         # Prepare target labels as text, e.g., "0" or "1"
#         if labels is not None:
#             encoded_labels = self.tokenizer(
#                 labels, max_length=2, truncation=True, 
#                 padding="max_length", return_tensors="pt"
#             ).input_ids.to(self.t5.device)
#         else:
#             encoded_labels = None

#         # Get the model output (logits for the sequence)
#         output = self.t5(
#             input_ids=encoded_input['input_ids'],
#             attention_mask=encoded_input['attention_mask'],
#             labels=encoded_labels
#         )

#         return output

# # Load the pretrained T5 model and tokenizer
# print("Loading T5 model and tokenizer...")
# t5_model = T5ForConditionalGeneration.from_pretrained('t5-small')
# tokenizer = T5Tokenizer.from_pretrained('t5-small')

# # Initialize the T5Sentinel model
# print("Initializing the T5Sentinel model...")
# t5_model = T5Sentinel(t5_model, tokenizer).to("cuda")

# # Define the loss function and optimizer
# criterion = nn.CrossEntropyLoss()
# optimizer = optim.Adam(t5_model.parameters(), lr=1e-4)

# # Set the number of epochs for training
# num_epochs = 10
# t5_model.train()

# print("Starting training...")
# for epoch in range(num_epochs):
#     running_loss = 0.0
#     correct_predictions = 0
#     total_predictions = 0

#     for texts, labels in train_loader:
#         # Convert texts to a list and move labels to GPU
#         texts = list(texts)
#         labels = [str(label) for label in labels]  # Convert integer labels to strings

#         # Zero the gradients
#         optimizer.zero_grad()
#         # Forward pass with the model
#         outputs = t5_model(texts, labels)
#         # Compute the loss using logits and labels
#         loss = outputs.loss
#         # Backward pass and optimization
#         loss.backward()
#         optimizer.step()

#         # Accumulate the loss
#         running_loss += loss.item()

#         # Generate predictions by decoding the output
#         generated_tokens = outputs.logits.argmax(dim=-1)
#         predictions = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)

#         # Compare predictions with labels for accuracy
#         correct_predictions += sum(p.strip() == l for p, l in zip(predictions, labels))
#         total_predictions += len(labels)

#     # Calculate average loss and accuracy for the epoch
#     epoch_loss = running_loss / len(train_loader)
#     epoch_accuracy = correct_predictions / total_predictions
#     print(f"Epoch {epoch+1}/{num_epochs}, Loss: {epoch_loss:.4f}, Accuracy: {epoch_accuracy:.4f}")

# print("Training complete")


# # Save the trained model to a file
# print("Saving the model...")
# torch.save(t5_model.state_dict(), 't5_sentinel.pth')
# print("Model saved as 't5_sentinel.pth'")


In [28]:
# t5_model.eval()

# # Evaluate the model on the new dataset
# def evaluate(model, dataloader, device):
#     model.eval()
#     all_preds = []
#     all_labels = []

#     with torch.no_grad():
#         for input_ids, attention_mask, labels in tqdm(dataloader, desc="Evaluating"):
#             input_ids, attention_mask, labels = input_ids.to(device), attention_mask.to(device), labels.to(device)

#             logits = model(input_ids, attention_mask)
#             preds = torch.argmax(logits, dim=1)

#             all_preds.extend(preds.cpu().numpy())
#             all_labels.extend(labels.cpu().numpy())

#     accuracy = accuracy_score(all_labels, all_preds)
#     f1 = f1_score(all_labels, all_preds, average='weighted')
#     report = classification_report(all_labels, all_preds)

#     return accuracy, f1, report

# # Run the evaluation
# print("Evaluating the model on the new dataset...")
# accuracy, f1, report = evaluate(t5_model, test_loader, device)

# # Print the evaluation results
# print(f"Accuracy: {accuracy:.4f}")
# print(f"F1 Score: {f1:.4f}")
# print("Classification Report:")
# print(report)

In [29]:
# import torch
# from sklearn.metrics import classification_report, accuracy_score, f1_score, confusion_matrix
# from torch.utils.data import DataLoader

# # Load the trained model
# print("Loading the trained T5 model...")
# t5_model = T5Sentinel(t5_model=T5ForConditionalGeneration.from_pretrained('t5-small'), tokenizer=T5Tokenizer.from_pretrained('t5-small'))
# t5_model.load_state_dict(torch.load('t5_sentinel.pth'))
# t5_model.to("cuda")
# t5_model.eval()

# # Create a DataLoader for the test dataset
# test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

# all_labels = []
# all_predictions = []

# with torch.no_grad():
#     print("Evaluating the model on the test dataset...")
#     for texts, labels in test_loader:
#         texts = list(texts)  # Convert to list if needed
#         labels = [str(label) for label in labels]  # Ensure labels are strings

#         # Tokenize and encode the input texts for the T5 model
#         encoded_input = t5_model.tokenizer(
#             texts, max_length=512, truncation=True,
#             padding="max_length", return_tensors="pt"
#         ).to(t5_model.t5.device)  # Move input to the same device as the model

#         # Get model predictions
#         outputs = t5_model(
#             input_ids=encoded_input['input_ids'],
#             attention_mask=encoded_input['attention_mask']
#         )

#         # Generate predictions by decoding the output
#         generated_tokens = outputs.logits.argmax(dim=-1)
#         predictions = t5_model.tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)

#         all_labels.extend(labels)  # Add true labels to the list
#         all_predictions.extend(predictions)  # Add predicted labels to the list

# # Generate the classification report
# print("Classification Report:")
# print(classification_report(all_labels, all_predictions))

# # Calculate accuracy and F1 score
# accuracy = accuracy_score(all_labels, all_predictions)
# f1 = f1_score(all_labels, all_predictions, average='weighted')
# cm = confusion_matrix(all_labels, all_predictions)
# tn, fp, fn, tp = cm.ravel() if cm.size == 4 else (0, 0, 0, 0)  # Handle binary classification

# # Calculate FNR and FPR
# fnr = fn / (fn + tp) if (fn + tp) > 0 else 0  # Avoid division by zero
# fpr = fp / (fp + tn) if (fp + tn) > 0 else 0  # Avoid division by zero

# # Print metrics
# print(f"Accuracy: {accuracy:.4f}")
# print(f"F1 Score: {f1:.4f}")
# print(f"False Negative Rate (FNR): {fnr:.4f}")
# print(f"False Positive Rate (FPR): {fpr:.4f}")


# T5 model on their dataset and testing on our dataset

In [30]:
# # ai_df = pd.read_csv("/kaggle/input/3500-datapoints/updated_dataset.csv", encoding='ISO-8859-1')
# # ai_df = ai_df.replace({'\r': '', '\n': ''}, regex=True)
# # filtered_df = ai_df[ai_df['ai_generated_text'].notna()]
# # human_df = pd.DataFrame(filtered_df['text'])
# # ai_df = pd.DataFrame(filtered_df['ai_generated_text'])
# # ai_df['ai_generated_text'] = ai_df['ai_generated_text'].apply(preprocess_text)
# # ai_df.rename(columns = {'ai_generated_text':'text'}, inplace = True)
# # human_df['label'] = 0
# # ai_df['label'] = 1
# # my_df = pd.concat([human_df[:1000], ai_df[:1000]], ignore_index=True)



# my_df=my_df[:2000]

# class TextDataset(Dataset):
#     def __init__(self, dataframe):
#         self.dataframe = dataframe

#     def __len__(self):
#         return len(self.dataframe)

#     def __getitem__(self, idx):
#         text = self.dataframe.iloc[idx, 0]  # assuming 'text' is the first column
#         label = int(self.dataframe.iloc[idx, 1])  # convert label to integer
#         return text, label

# # Data Loaders
# # train_dataset = TextDataset(train_df)
# test_dataset = TextDataset(my_df)

# # train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)  # Single datapoint per batch
# test_loader = DataLoader(test_dataset, batch_size=2, shuffle=False)

In [31]:
# import torch
# from torch.utils.data import DataLoader
# from sklearn.metrics import classification_report

# # Load the saved model weights
# # ensemble_model.load_state_dict(torch.load('/kaggle/working/t5_sentinel.pth'))
# t5_model.eval()

# # DataLoader for the test dataset
# # test_loader = DataLoader(df_HC3, batch_size=2, shuffle=False)

# # Function to test the model on the test data
# # Evaluate the model on the new dataset
# print("Evaluating the T5 model on the our dataset...")
# all_labels = []
# all_predictions = []

# with torch.no_grad():
#     for texts, labels in test_loader:
#         texts = list(texts)
#         labels = labels.to("cuda")

#         outputs = t5_model(texts)
#         _, predicted = torch.max(outputs, 1)

#         all_labels.extend(labels.cpu().numpy())
#         all_predictions.extend(predicted.cpu().numpy())

# # Classification report
# print("Classification Report:")
# print(classification_report(all_labels, all_predictions))

# # Calculate F1 Score, FNR, and FPR
# f1 = f1_score(all_labels, all_predictions, average='weighted')
# cm = confusion_matrix(all_labels, all_predictions)
# tn, fp, fn, tp = cm.ravel()
# fnr = fn / (fn + tp)
# fpr = fp / (fp + tn)

# print(f"F1 Score: {f1:.4f}")
# print(f"False Negative Rate (FNR): {fnr:.4f}")
# print(f"False Positive Rate (FPR): {fpr:.4f}")



# T5 model trained on their dataset and testing on HC3 dataset

In [32]:
# # # !pip install datasets
# # from datasets import load_dataset

# # ds = load_dataset("Hello-SimpleAI/HC3", "all")

# # # Concatenate inner lists into a single string for each element
# # concatenated = [' '.join(text_list) for text_list in ds['train']['human_answers']]

# # # Create a DataFrame with a single column named 'text'
# # df_human_hc3 = pd.DataFrame(concatenated, columns=['text'])
# # df_human_hc3['label'] = 0
# # df_human_hc3 = df_human_hc3[:1000]


# # concatenated_ = [' '.join(text_list) for text_list in ds['train']['chatgpt_answers']]

# # # Create a DataFrame with a single column named 'text'
# # df_ai_hc3 = pd.DataFrame(concatenated_, columns=['text'])
# # df_ai_hc3['label'] = 1
# # df_ai_hc3 = df_ai_hc3[:1000]



# # df_HC3 = pd.concat([df_ai_hc3, df_human_hc3], ignore_index=True)
# # # df_HC3 = df_HC3[:100]

# class TextDataset(Dataset):
#     def __init__(self, dataframe):
#         self.dataframe = dataframe

#     def __len__(self):
#         return len(self.dataframe)

#     def __getitem__(self, idx):
#         text = self.dataframe.iloc[idx, 0]  # assuming 'text' is the first column
#         label = int(self.dataframe.iloc[idx, 1])  # convert label to integer
#         return text, label

# # Data Loaders
# # train_dataset = TextDataset(train_df)
# test_dataset = TextDataset(df_HC3)

# # train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)  # Single datapoint per batch
# test_loader = DataLoader(test_dataset, batch_size=2, shuffle=False)

In [33]:
# import torch
# from torch.utils.data import DataLoader
# from sklearn.metrics import classification_report

# # Load the saved model weights
# # ensemble_model.load_state_dict(torch.load('/kaggle/input/2_epoch_mix_model/other/default/1/our_ensemble_model.pth'))
# t5_model.eval()

# # DataLoader for the test dataset
# # test_loader = DataLoader(df_HC3, batch_size=2, shuffle=False)

# # Evaluate the model on the new dataset
# print("Evaluating the T5 model on the HC3 dataset...")
# all_labels = []
# all_predictions = []

# with torch.no_grad():
#     for texts, labels in test_loader:
#         texts = list(texts)
#         labels = labels.to("cuda")

#         outputs = t5_model(texts)
#         _, predicted = torch.max(outputs, 1)

#         all_labels.extend(labels.cpu().numpy())
#         all_predictions.extend(predicted.cpu().numpy())

# # Classification report
# print("Classification Report:")
# print(classification_report(all_labels, all_predictions))

# # Calculate F1 Score, FNR, and FPR
# f1 = f1_score(all_labels, all_predictions, average='weighted')
# cm = confusion_matrix(all_labels, all_predictions)
# tn, fp, fn, tp = cm.ravel()
# fnr = fn / (fn + tp)
# fpr = fp / (fp + tn)

# print(f"F1 Score: {f1:.4f}")
# print(f"False Negative Rate (FNR): {fnr:.4f}")
# print(f"False Positive Rate (FPR): {fpr:.4f}")



# Our code

### Mukund's code

In [ ]:
# import torch
# import torch.nn as nn
# import torch.optim as optim
# from transformers import RobertaModel, RobertaTokenizer, T5ForConditionalGeneration, T5Tokenizer
# from sklearn.model_selection import train_test_split
# from torch.utils.data import DataLoader, Dataset
# from tqdm import tqdm

# # Assuming 'df' is your dataset with columns ['text', 'label']
# train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

# # Dataset Class
# class TextDataset(Dataset):
#     def __init__(self, dataframe):
#         self.dataframe = dataframe

#     def __len__(self):
#         return len(self.dataframe)

#     def __getitem__(self, idx):
#         text = self.dataframe.iloc[idx, 0]  # assuming 'text' is the first column
#         label = int(self.dataframe.iloc[idx, 1])  # convert label to integer
#         return text, label

# # Data Loaders
# train_dataset = TextDataset(train_df)
# test_dataset = TextDataset(test_df)

# train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)  # Single datapoint per batch
# test_loader = DataLoader(test_dataset, batch_size=2, shuffle=False)

# # Function to process chunks and compute weighted probabilities for both models
# def process_chunks(model, tokenizer, texts, classifier, name, max_length=512, max_chunks=4):
#     encoded_texts = tokenizer(
#         texts,
#         truncation=False,
#         padding=True,
#         return_tensors="pt"
#     )

#     input_ids = encoded_texts['input_ids'].cuda()
#     attention_mask = encoded_texts['attention_mask'].cuda()

#     batch_probs = []
#     for i in range(len(input_ids)):
#         input_id = input_ids[i]
#         attention_mask_i = attention_mask[i]
        
#         # print(f"Input text {i} for {name}: {texts[i]}")
        
#         print(f"Processing text {i} with input IDs and attention mask for {name}")
# #         print(f"Text {i}: input IDs : {input_id}")

#         print(f"Text {i}: input IDs shape: {input_id.shape}")

#         # Split the input into chunks of 512 tokens
#         chunks = [input_id[j:j + 512] for j in range(0, len(input_id), 512)]
#         attention_chunks = [attention_mask_i[j:j + 512] for j in range(0, len(input_id), 512)]

#         print(f"Text {i}: number of chunks: {len(chunks)}")  # Debugging line
        
#         if len(chunks) > max_chunks:
#             chunks = chunks[:max_chunks]
#             attention_chunks = attention_chunks[:max_chunks]

#         last_chunk_size = len(chunks[-1])
#         print(f"Last chunk size for text {i}: {last_chunk_size}")  # Debugging line
        
#         if last_chunk_size < 512:
#             padding_length = 512 - last_chunk_size
#             chunks[-1] = torch.cat([chunks[-1], torch.zeros(padding_length, dtype=torch.long).cuda()])
#             attention_chunks[-1] = torch.cat([attention_chunks[-1], torch.zeros(padding_length, dtype=torch.long).cuda()])

#         chunk_probs = []
#         for chunk, mask in zip(chunks, attention_chunks):
#             chunk = chunk.unsqueeze(0).cuda()
#             mask = mask.unsqueeze(0).cuda()

#             # print(f"Getting output from {name} for chunk...")

#             # Get model outputs
#             outputs = model(input_ids=chunk, attention_mask=mask)
#             cls_output = outputs.last_hidden_state[:, 0, :]
#             # print(f"CLS output shape for {name} chunk: {cls_output.shape}")

#             logits = classifier(cls_output)
#             # print(f"Logits from classifier for {name} chunk: {logits}")

#             probabilities = torch.sigmoid(logits)
#             # print(f"Probabilities for {name} chunk: {probabilities}")

#             chunk_probs.append(probabilities)

#         chunk_probs = torch.cat(chunk_probs, dim=0)  # Shape: (num_chunks, 1)
#         # print(f"All chunk probabilities for {name} text {i}: {chunk_probs}")

#         chunk_weights = torch.ones(len(chunks), dtype=torch.float).cuda() * 512
#         chunk_weights[-1] = last_chunk_size
#         chunk_weights = chunk_weights / 512
#         # print(f"Chunk weights for {name} text {i}: {chunk_weights}")

#         weighted_probs = torch.sum(chunk_probs.squeeze() * chunk_weights) / torch.sum(chunk_weights)
#         # print(f"Weighted average probability for {name} text {i}: {weighted_probs}")

#         batch_probs.append(weighted_probs)

#     final_probs = torch.stack(batch_probs)  # Shape: (batch_size,)
#     # print(f"Final probabilities for {name}: {final_probs}")
    
#     return final_probs

# # RoBERTa Model with classifier
# class RobertaSentinel(nn.Module):
#     def __init__(self):
#         super(RobertaSentinel, self).__init__()
#         self.roberta = RobertaModel.from_pretrained('roberta-base')
#         self.classifier = nn.Sequential(
#             nn.Linear(768, 768),
#             nn.ReLU(),
#             nn.Dropout(0.2),
#             nn.Linear(768, 512),
#             nn.ReLU(),
#             nn.Dropout(0.2),
#             nn.Linear(512, 64),
#             nn.ReLU(),
#             nn.Linear(64, 1)
#         )

#     def forward(self, texts, tokenizer):
#         probabilities = process_chunks(self.roberta, tokenizer, texts, self.classifier, 'roberta')
#         return probabilities

# # T5 Model with classifier
# class T5BinaryClassifier(nn.Module):
#     def __init__(self, model_name="t5-small"):
#         super(T5BinaryClassifier, self).__init__()
#         self.t5 = T5ForConditionalGeneration.from_pretrained(model_name)
#         self.classifier = nn.Linear(self.t5.config.d_model, 1)

#     def forward(self, texts, tokenizer):
#         probabilities = process_chunks(self.t5.encoder, tokenizer, texts, self.classifier, 'T5')
#         return probabilities

# # Initialize models and tokenizers
# roberta_model = RobertaSentinel().cuda()
# roberta_tokenizer = RobertaTokenizer.from_pretrained('roberta-base')

# t5_model = T5BinaryClassifier().cuda()
# t5_tokenizer = T5Tokenizer.from_pretrained('t5-small')

# # Ensemble model class
# class EnsembleModel(nn.Module):
#     def __init__(self, roberta_model, t5_model, roberta_tokenizer, t5_tokenizer):
#         super(EnsembleModel, self).__init__()
#         self.roberta_model = roberta_model
#         self.t5_model = t5_model
#         self.roberta_tokenizer = roberta_tokenizer
#         self.t5_tokenizer = t5_tokenizer

#     def forward(self, texts):
#         # Get probabilities from both models
# #         print(f"texts is {texts}")
#         roberta_probs = self.roberta_model(texts, self.roberta_tokenizer)
#         print(f"RoBERTa probabilities: {roberta_probs}")
#         print(f"roberta part over")
        
        
#         # print(f"texts is {texts}")
#         t5_probs = self.t5_model(texts, self.t5_tokenizer)
#         print(f"T5 probabilities: {t5_probs}")
#         print(f"T5 part over")

#         # Assign weights for ensembling
#         roberta_weight = 0.5
#         t5_weight = 0.5
#         print(f"Weights: RoBERTa = {roberta_weight}, T5 = {t5_weight}")

# #         Compute weighted probabilities
#         weighted_probs = (roberta_weight * roberta_probs) + (t5_weight * t5_probs)
#         print(f"Weighted probabilities: {weighted_probs}")

#         return roberta_probs,t5_probs,weighted_probs

# # Initialize the ensemble model
# ensemble_model = EnsembleModel(roberta_model, t5_model, roberta_tokenizer, t5_tokenizer).cuda()

# # Loss and optimizer
# criterion = nn.BCEWithLogitsLoss()
# optimizer = optim.Adam(ensemble_model.parameters(), lr=1e-4)

# # Training loop
# num_epochs = 5
# ensemble_model.train()

# for epoch in range(num_epochs):
#     running_loss = 0.0
#     correct_predictions = 0
#     total_predictions = 0

#     for texts, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}"):
#         texts = list(texts)  # Convert batch of texts to a list
#         labels = labels.to("cuda").float()

#         labels = labels.view(-1, 1)

#         optimizer.zero_grad()

#         # Forward pass through the ensemble model
#         roberta_probs,t5_probs,final_probs = ensemble_model(texts)

#         final_probs = final_probs.view(-1, 1)

#         # Compute the loss
#         loss = criterion(final_probs, labels)

#         # Backward pass and optimization
#         loss.backward()
#         optimizer.step()

#         running_loss += loss.item()
#         ones = [roberta_probs,t5_probs]
#         zeros = [1 - roberta_probs,1 - t5_probs]
        
#         max_ones = torch.max(ones[0], ones[1])  # Find max between (1 - roberta_probs) and (1 - t5_probs)
#         max_zeros = torch.max(zeros[0], zeros[1])  # Find max between roberta_probs and t5_probs
#         maxi = torch.max(max_ones,max_zeros)
#         predicted = (max_ones > 0.5).float()
#         correct_predictions += (predicted == labels).sum().item()
#         total_predictions += labels.size(0)

#         # Debugging logs
#         for i in range(labels.size(0)):
#             actual_class = labels[i].item()
#             final_prob = final_probs[i].item()
#             print(f"data point {i+1} : Actual Class: {actual_class}")
#             print(f"From Roberta for class 1 : {roberta_probs[i]}")
#             print(f"From Roberta for class 0 : { 1 - roberta_probs[i]}")
#             print(f"From T5 for class 1 : {t5_probs[i]}")
#             print(f"From T5 for class 0 : { 1 - t5_probs[i]}")
#             print(f"predicted class is : {predicted[i]}")
#             print(f"probability choosen is: {maxi[i]}")
            
# #             print(f"Data Point {i+1}: Actual Class: {actual_class}, Final Probability for class 1: {final_prob:.4f}, Prediction: {predicted[i].item()}")

#     epoch_loss = running_loss / len(train_loader)
#     epoch_accuracy = correct_predictions / total_predictions
#     print(f"Epoch {epoch+1}, Loss: {epoch_loss:.4f}, Accuracy: {epoch_accuracy:.4f}")

### Himanshu's code

#### 1. Probabilties + logits 
* you don't have to run it. !!! Danger

In [ ]:
# import torch
# import torch.nn as nn
# import torch.optim as optim
# from transformers import  T5ForConditionalGeneration, T5Tokenizer
# from transformers import DistilBertModel, DistilBertTokenizer

# from sklearn.model_selection import train_test_split
# from torch.utils.data import DataLoader, Dataset
# from tqdm import tqdm

# # Assuming 'df' is your dataset with columns ['text', 'label']
# train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

# # Dataset Class
# class TextDataset(Dataset):
#     def __init__(self, dataframe):
#         self.dataframe = dataframe

#     def __len__(self):
#         return len(self.dataframe)

#     def __getitem__(self, idx):
#         text = self.dataframe.iloc[idx, 0]  # assuming 'text' is the first column
#         label = int(self.dataframe.iloc[idx, 1])  # convert label to integer
#         return text, label

# # Data Loaders
# train_dataset = TextDataset(train_df)
# test_dataset = TextDataset(test_df)

# train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)  # Single datapoint per batch
# test_loader = DataLoader(test_dataset, batch_size=2, shuffle=False)

# # Function to process chunks and compute weighted probabilities for both models
# def process_chunks(model, tokenizer, texts, classifier, name, max_length=512, max_chunks=4):
#     encoded_texts = tokenizer(
#         texts,
#         truncation=False,
#         padding=True,
#         return_tensors="pt"
#     )

#     input_ids = encoded_texts['input_ids'].cuda()
#     attention_mask = encoded_texts['attention_mask'].cuda()

#     batch_probs = []
#     batch_logits = []
#     for i in range(len(input_ids)):
#         input_id = input_ids[i]
#         attention_mask_i = attention_mask[i]
        
#         # print(f"Input text {i} for {name}: {texts[i]}")
        
#         print(f"Processing text {i} with input IDs and attention mask for {name}")
# #         print(f"Text {i}: input IDs : {input_id}")

#         print(f"Text {i}: input IDs shape: {input_id.shape}")

#         # Split the input into chunks of 512 tokens
#         chunks = [input_id[j:j + 512] for j in range(0, len(input_id), 512)]
#         attention_chunks = [attention_mask_i[j:j + 512] for j in range(0, len(input_id), 512)]

#         print(f"Text {i}: number of chunks: {len(chunks)}")  # Debugging line
        
#         if len(chunks) > max_chunks:
#             chunks = chunks[:max_chunks]
#             attention_chunks = attention_chunks[:max_chunks]

#         last_chunk_size = len(chunks[-1])
#         print(f"Last chunk size for text {i}: {last_chunk_size}")  # Debugging line

#         if last_chunk_size < 512:
#             padding_length = 512 - last_chunk_size
#             chunks[-1] = torch.cat([chunks[-1], torch.zeros(padding_length, dtype=torch.long).cuda()])
#             attention_chunks[-1] = torch.cat([attention_chunks[-1], torch.zeros(padding_length, dtype=torch.long).cuda()])

#         chunk_probs = []
#         chunk_logits= []
#         for chunk, mask in zip(chunks, attention_chunks):
#             chunk = chunk.unsqueeze(0).cuda()
#             mask = mask.unsqueeze(0).cuda()

#             # print(f"Getting output from {name} for chunk...")

#             # Get model outputs
#             outputs = model(input_ids=chunk, attention_mask=mask)
#             cls_output = outputs.last_hidden_state[:, 0, :]
#             # print(f"CLS output shape for {name} chunk: {cls_output.shape}")

#             logits = classifier(cls_output)
#             chunk_logits.append(logits)
#             # print(f"Logits from classifier for {name} chunk: {logits}")

#             probabilities = torch.sigmoid(logits)
#             # print(f"Probabilities for {name} chunk: {probabilities}")
#             chunk_probs.append(probabilities)

#         chunk_probs = torch.cat(chunk_probs, dim=0)  # Shape: (num_chunks, 1)
#         print(f"All chunk probabilities for {name} text {i}: {chunk_probs}")

#         chunk_logits = torch.cat(chunk_logits, dim=0)
#         # print(f"All chunk logits for {name} text {i}: {chunk_logits}")


#         chunk_weights = torch.ones(len(chunks), dtype=torch.float).cuda() * 512
#         chunk_weights[-1] = last_chunk_size
#         chunk_weights = chunk_weights / 512
#         # print(f"Chunk weights for {name} text {i}: {chunk_weights}")
#         # calculate weighted probabilities
#         weighted_probs = torch.sum(chunk_probs.squeeze() * chunk_weights) / torch.sum(chunk_weights)
#         # print(f"Weighted probability for {name} text {i} : {weighted_probs}")
        
#         #calculate weighted logits(only use if needed other wise just use  mean of all logits)
#         weighted_logits = torch.sum(chunk_logits.squeeze() * chunk_weights) / torch.sum(chunk_weights)
#         ## weighted_logits = torch.mean(torch.stack(chunk_logits), dim=0)

#         batch_probs.append(weighted_probs)
#         batch_logits.append(weighted_logits)

#     final_probs = torch.stack(batch_probs)  # Shape: (batch_size,)
#     final_logits = torch.stack(batch_logits)
#     print(f"Final probabilities for {name}: {final_probs}")
    
#     return final_probs, final_logits

# # RoBERTa Model with classifier
# class DistilSentinel(nn.Module):
#     def __init__(self):
#         super(DistilSentinel, self).__init__()
#         self.distil = DistilBertModel.from_pretrained('distilbert-base-uncased')
#         self.classifier = nn.Sequential(
#             nn.Linear(768, 768),
#             nn.GELU(),
#             nn.Dropout(0.2),
#             nn.Linear(768, 1)
#         )

#     def forward(self, texts, tokenizer):
#         probabilities, logits = process_chunks(self.distil, tokenizer, texts, self.classifier, 'distilbert')
#         return probabilities, logits

# # T5 Model with classifier
# class T5BinaryClassifier(nn.Module):
#     def __init__(self, model_name="t5-small"):
#         super(T5BinaryClassifier, self).__init__()
#         self.t5 = T5ForConditionalGeneration.from_pretrained(model_name)
#         self.classifier = nn.Linear(self.t5.config.d_model, 1)

#     def forward(self, texts, tokenizer):
#         probabilities, logits = process_chunks(self.t5.encoder, tokenizer, texts, self.classifier, 'T5')
#         return probabilities, logits

# # Initialize models and tokenizers
# distil_model = DistilSentinel().cuda()
# distil_tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

# t5_model = T5BinaryClassifier().cuda()
# t5_tokenizer = T5Tokenizer.from_pretrained('t5-small')

# # Ensemble model class
# class EnsembleModel(nn.Module):
#     def __init__(self, distil_model, t5_model, distil_tokenizer, t5_tokenizer):
#         super(EnsembleModel, self).__init__()
#         self.distil_model = distil_model
#         self.t5_model = t5_model
#         self.distil_tokenizer = distil_tokenizer
#         self.t5_tokenizer = t5_tokenizer

#     def forward(self, texts):
#         # Get probabilities from both models
# #         print(f"texts is {texts}")
#         distil_probs, distil_logits = self.distil_model(texts, self.distil_tokenizer)
#         print(f"Distil probabilities: {distil_probs}")
#         # print(f"Distil logits: {distil_logits}")
#         print(f"Distiil part over")
        
        
#         # print(f"texts is {texts}")
#         t5_probs, t5_logits = self.t5_model(texts, self.t5_tokenizer)
#         print(f"T5 probabilities: {t5_probs}")
#         # print(f"T5 logits: {t5_logits}")
#         print(f"T5 part over")

#         # abb btao logits and probabs kak= kya krna hae

#         # weighted probs ki jrurt hin nhi hae kyuki isee hum loss mae use hinn nhi kr rhe. humiske jagah pe logitsuse krenge
# #         # Assign weights for ensembling
# #         distil_weight = 0.5
# #         t5_weight = 0.5
# #         print(f"Weights: distil = {distil_weight}, T5 = {t5_weight}")

# # #         Compute weighted probabilities
# #         weighted_probs = (distil_weight * distil_probs) + (t5_weight * t5_probs)
# #         print(f"Weighted probabilities: {weighted_probs}")

#         # here we have two options to check for final_logits
#         #1. take average of both logits. we can't take weighted because we want equal contribution from each model.
#         #2. we can try max logits in ditil and t5 logits

#         #Note: try one of the option at a time.
#         #1st method:
#         # final_logits = (distil_logits + t5_logits) / 2

#         #2nd method:
#         final_logits = torch.max(distil_logits, t5_logits)


#         return distil_probs,t5_probs,final_logits

# # Initialize the ensemble model
# ensemble_model = EnsembleModel(distil_model, t5_model, distil_tokenizer, t5_tokenizer).cuda()

# # Loss and optimizer
# criterion = nn.BCEWithLogitsLoss()
# optimizer = optim.Adam(ensemble_model.parameters(), lr=1e-4)

# # Training loop
# num_epochs = 5
# ensemble_model.train()

# for epoch in range(num_epochs):
#     running_loss = 0.0
#     correct_predictions = 0
#     total_predictions = 0

#     for texts, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}"):
#         texts = list(texts)  # Convert batch of texts to a list
#         labels = labels.to("cuda").float()

#         labels = labels.view(-1, 1)

#         optimizer.zero_grad()

#         # Forward pass through the ensemble model
#         distil_probs,t5_probs,final_logits = ensemble_model(texts)

#         final_logits = final_logits.view(-1, 1)
#         # Ensure that labels and logits have the same shape
#         print(f"Labels shape: {labels.shape}, Final logits shape: {final_logits.shape}")

#         # Compute the loss
#         loss = criterion(final_logits, labels)

#         # Backward pass and optimization
#         loss.backward()
#         optimizer.step()

#         running_loss += loss.item()

#         # yeha pe check krna hae ki isme kitni probabs ki value ayegi
#         ones = [distil_probs, t5_probs]
#         zeros = [1 - distil_probs,1 - t5_probs]
#         print(f"ones probability values: {ones}")
#         print(f"zeros probability values: {zeros}")

        
#         max_ones = torch.max(ones[0], ones[1])  # Find max between (1 - roberta_probs) and (1 - t5_probs)
#         max_zeros = torch.max(zeros[0], zeros[1])  # Find max between roberta_probs and t5_probs

#         print(f"MaxOnes values: {max_ones}")
#         print(f"MaxZeros values: {max_zeros}")

#         maxi = torch.max(max_ones,max_zeros)
#         print(f"Maximum value: {maxi}")
#         # predicted = (max_ones > 0.5).float() # i don't know why we are taking max_ones. we should take maxi at place of max_ones
#         #predicted = (maxi > 0.5).float() # this is also bullshit man. 

#         # Compare max_ones and max_zeros to determine the predicted class.
#         # If max_ones > max_zeros, predict class 1; otherwise, predict class 0.
#         predicted = (max_ones > max_zeros).float()
        
#         correct_predictions += (predicted == labels).sum().item()
#         total_predictions += labels.size(0)

#         # Debugging logs
#         for i in range(labels.size(0)):
#             actual_class = labels[i].item()
#             # final_prob = final_probs[i].item()
#             print(f"data point {i+1} : Actual Class: {actual_class}")
#             print(f"From distil for class 1 : {distil_probs[i]}")
#             print(f"From distil for class 0 : { 1 - distil_probs[i]}")
#             print(f"From T5 for class 1 : {t5_probs[i]}")
#             print(f"From T5 for class 0 : { 1 - t5_probs[i]}")
#             print(f"predicted class is : {predicted[i]}")
#             print(f"probability choosen is: {maxi[i]}")
            
# #             print(f"Data Point {i+1}: Actual Class: {actual_class}, Final Probability for class 1: {final_prob:.4f}, Prediction: {predicted[i].item()}")

#     epoch_loss = running_loss / len(train_loader)
#     epoch_accuracy = correct_predictions / total_predictions
#     print(f"Epoch {epoch+1}, Loss: {epoch_loss:.4f}, Accuracy: {epoch_accuracy:.4f}")


## Our Final Architecture

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from transformers import T5ForConditionalGeneration, T5Tokenizer
from transformers import DistilBertModel, DistilBertTokenizer

from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm

# Assuming 'df' is your dataset with columns ['text', 'label']
train_df, test_df = train_test_split(my_df, test_size=0.2, random_state=42)

# Dataset Class
class TextDataset(Dataset):
    def __init__(self, dataframe):
        self.dataframe = dataframe

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        text = self.dataframe.iloc[idx, 0]  # assuming 'text' is the first column
        label = int(self.dataframe.iloc[idx, 1])  # convert label to integer
        return text, label

# Data Loaders
train_dataset = TextDataset(train_df)
test_dataset = TextDataset(test_df)

train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)  # Single datapoint per batch
test_loader = DataLoader(test_dataset, batch_size=2, shuffle=False)

# Function to process chunks and compute logits for both models
def process_chunks(model, tokenizer, texts, classifier, name, max_length=512, max_chunks=4):
    encoded_texts = tokenizer(
        texts,
        truncation=False,
        padding=True,
        return_tensors="pt"
    )

    input_ids = encoded_texts['input_ids'].cuda()
    attention_mask = encoded_texts['attention_mask'].cuda()

    batch_logits = []
    for i in range(len(input_ids)):
        input_id = input_ids[i]
        attention_mask_i = attention_mask[i]

        # Split the input into chunks of 512 tokens
        chunks = [input_id[j:j + 512] for j in range(0, len(input_id), 512)]
        attention_chunks = [attention_mask_i[j:j + 512] for j in range(0, len(input_id), 512)]

        if len(chunks) > max_chunks:
            chunks = chunks[:max_chunks]
            attention_chunks = attention_chunks[:max_chunks]

        last_chunk_size = len(chunks[-1])

        if last_chunk_size < 512:
            padding_length = 512 - last_chunk_size
            chunks[-1] = torch.cat([chunks[-1], torch.zeros(padding_length, dtype=torch.long).cuda()])
            attention_chunks[-1] = torch.cat([attention_chunks[-1], torch.zeros(padding_length, dtype=torch.long).cuda()])

        chunk_logits = []
        for chunk, mask in zip(chunks, attention_chunks):
            chunk = chunk.unsqueeze(0).cuda()
            mask = mask.unsqueeze(0).cuda()

            # Get model outputs
            outputs = model(input_ids=chunk, attention_mask=mask)
            cls_output = outputs.last_hidden_state[:, 0, :]

            logits = classifier(cls_output)
            chunk_logits.append(logits)

        chunk_logits = torch.cat(chunk_logits, dim=0)  # Shape: (num_chunks, 1)

        # Calculate weights based on chunk size
        chunk_weights = torch.ones(len(chunks), dtype=torch.float).cuda() * 512
        chunk_weights[-1] = last_chunk_size
        chunk_weights = chunk_weights / 512

        # Compute weighted logits
        weighted_logits = torch.sum(chunk_logits.squeeze() * chunk_weights) / torch.sum(chunk_weights)

        batch_logits.append(weighted_logits)

    final_logits = torch.stack(batch_logits)  # Shape: (batch_size,)
    return final_logits

# RoBERTa Model with classifier
class DistilSentinel(nn.Module):
    def __init__(self):
        super(DistilSentinel, self).__init__()
        self.distil = DistilBertModel.from_pretrained('distilbert-base-uncased')
        self.classifier = nn.Sequential(
            nn.Linear(768, 768),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(768, 1)
        )

    def forward(self, texts, tokenizer):
        logits = process_chunks(self.distil, tokenizer, texts, self.classifier, 'distilbert')
        return logits

# T5 Model with classifier
class T5BinaryClassifier(nn.Module):
    def __init__(self, model_name="t5-small"):
        super(T5BinaryClassifier, self).__init__()
        self.t5 = T5ForConditionalGeneration.from_pretrained(model_name)
        self.classifier = nn.Linear(self.t5.config.d_model, 1)

    def forward(self, texts, tokenizer):
        logits = process_chunks(self.t5.encoder, tokenizer, texts, self.classifier, 'T5')
        return logits

# Initialize models and tokenizers
distil_model = DistilSentinel().cuda()
distil_tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

t5_model = T5BinaryClassifier().cuda()
t5_tokenizer = T5Tokenizer.from_pretrained('t5-small')

# Ensemble model class
class EnsembleModel(nn.Module):
    def __init__(self, distil_model, t5_model, distil_tokenizer, t5_tokenizer):
        super(EnsembleModel, self).__init__()
        self.distil_model = distil_model
        self.t5_model = t5_model
        self.distil_tokenizer = distil_tokenizer
        self.t5_tokenizer = t5_tokenizer

    def forward(self, texts):
        # Get logits from both models
        distil_logits = self.distil_model(texts, self.distil_tokenizer)
        t5_logits = self.t5_model(texts, self.t5_tokenizer)

        # Compute final logits as average
        final_logits = torch.max(distil_logits, t5_logits)

        return distil_logits, t5_logits, final_logits

# Initialize the ensemble model
ensemble_model = EnsembleModel(distil_model, t5_model, distil_tokenizer, t5_tokenizer).cuda()

# Loss and optimizer
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(ensemble_model.parameters(), lr=1e-4)

# Training loop
num_epochs = 1
ensemble_model.train()

for epoch in range(num_epochs):
    running_loss = 0.0
    correct_predictions = 0
    total_predictions = 0

    for texts, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}"):
        texts = list(texts)  # Convert batch of texts to a list
        labels = labels.to("cuda").float()

        labels = labels.view(-1, 1)

        optimizer.zero_grad()

        # Forward pass through the ensemble model
        distil_logits, t5_logits, final_logits = ensemble_model(texts)
        final_logits = final_logits.view(-1, 1)
        # Compute the loss
        loss = criterion(final_logits, labels)

        # Backward pass and optimization
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        
        # Make predictions
        predicted = (final_logits > 0).float()
        correct_predictions += (predicted == labels).sum().item()
        total_predictions += labels.size(0)

        # Debugging logs
        for i in range(labels.size(0)):
            actual_class = labels[i].item()
            final_prob = torch.sigmoid(final_logits[i]).item()
            print(f"Data point {i+1}: Actual Class: {actual_class}, Final Logit: {final_logits[i].item()}, Predicted Probability: {final_prob:.4f}, Prediction: {predicted[i].item()}")

    epoch_loss = running_loss / len(train_loader)
    epoch_accuracy = correct_predictions / total_predictions
    print(f"Epoch {epoch+1}, Loss: {epoch_loss:.4f}, Accuracy: {epoch_accuracy:.4f}")
# Save the model at the end of each epoch
print("Saving the ensemble model")
torch.save(ensemble_model.state_dict(), f'our_ensemble_model.pth')
print(f"Model saved as 'ensemble_model_epoch.pth'")

In [ ]:
from sklearn.metrics import classification_report, accuracy_score

# Evaluation function
def evaluate_model(model, test_loader, criterion):
    model.eval()  # Set the model to evaluation mode
    all_labels = []
    all_predictions = []
    total_loss = 0.0

    with torch.no_grad():
        for texts, labels in tqdm(test_loader, desc="Evaluating"):
            texts = list(texts)  # Convert batch of texts to a list
            labels = labels.to("cuda").float().view(-1, 1)

            # Forward pass through the ensemble model
            distil_logits, t5_logits, final_logits = model(texts)
            final_logits = final_logits.view(-1, 1)

            # Compute the loss
            loss = criterion(final_logits, labels)
            total_loss += loss.item()

            # Make predictions
            predictions = (final_logits > 0).float()

            # Store the true labels and predictions
            all_labels.extend(labels.cpu().numpy())
            all_predictions.extend(predictions.cpu().numpy())

    # Compute average loss over the test set
    avg_loss = total_loss / len(test_loader)

    # Compute accuracy
    accuracy = accuracy_score(all_labels, all_predictions)

    # Generate classification report
    report = classification_report(all_labels, all_predictions, target_names=["Class 0", "Class 1"])

    print(f"Test Loss: {avg_loss:.4f}")
    print(f"Test Accuracy: {accuracy:.4f}")
    print("Classification Report:")
    print(report)

    return avg_loss, accuracy, report

# Call the evaluation function
criterion = nn.BCEWithLogitsLoss()
test_loss, test_accuracy, classification_report = evaluate_model(ensemble_model, test_loader, criterion)

# Combining base paper dataset and our dataset.

In [34]:
# Define a function to preprocess each text
def preprocess_text(text):
    # Find the index of the first colon
    colon_index = text.find(':')
    # If a colon exists and is within the first 20 words, trim the text
    if colon_index != -1:
        # Get the words up to the colon's position
        words_before_colon = text[:colon_index].split()
        # Check if the colon is within the first 20 words
        if len(words_before_colon) <= 20:
            # Return the text after the colon, trimmed of leading/trailing spaces
            return text[colon_index + 1:].strip()
    # If no colon or colon is outside the first 20 words, return the original text
    return text



In [35]:
import pandas as pd

# Load the dataset
paper_dataframe = pd.read_csv('/kaggle/input/humanllm/Final_dataset.csv')

# Replace '\r' and '\n' with an empty string in all columns
paper_dataframe = paper_dataframe.replace({'\r': '', '\n': ''}, regex=True)


paper_dataframe.drop(columns=['uid'], inplace=True)

paper_dataframe = paper_dataframe[:5000]


ai_df = pd.read_csv("/kaggle/input/3500-datapoints/updated_dataset.csv", encoding='ISO-8859-1')
ai_df = ai_df.replace({'\r': '', '\n': ''}, regex=True)
filtered_df = ai_df[ai_df['ai_generated_text'].notna()]
human_df = pd.DataFrame(filtered_df['text'])
ai_df = pd.DataFrame(filtered_df['ai_generated_text'])
ai_df['ai_generated_text'] = ai_df['ai_generated_text'].apply(preprocess_text)
ai_df.rename(columns = {'ai_generated_text':'text'}, inplace = True)
human_df['label'] = 0
ai_df['label'] = 1
my_df = pd.concat([human_df, ai_df], ignore_index=True)
my_dataset = my_df.sample(frac=1, random_state=42).reset_index(drop=True)


my_dataset = my_dataset[:5000]


new_dataset = pd.concat([my_dataset, paper_dataframe], ignore_index = True)
dataset = new_dataset.sample(frac=1, random_state=42).reset_index(drop=True)


In [36]:
import torch
import torch.nn as nn
import torch.optim as optim
from transformers import T5ForConditionalGeneration, T5Tokenizer
from transformers import DistilBertModel, DistilBertTokenizer

from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm

# Assuming 'df' is your dataset with columns ['text', 'label']
train_df, test_df = train_test_split(dataset, test_size=0.2, random_state=42)

# Dataset Class
class TextDataset(Dataset):
    def __init__(self, dataframe):
        self.dataframe = dataframe

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        text = self.dataframe.iloc[idx, 0]  # assuming 'text' is the first column
        label = int(self.dataframe.iloc[idx, 1])  # convert label to integer
        return text, label

# Data Loaders
train_dataset = TextDataset(train_df)
test_dataset = TextDataset(test_df)

train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)  # Single datapoint per batch
test_loader = DataLoader(test_dataset, batch_size=2, shuffle=False)

# Function to process chunks and compute logits for both models
def process_chunks(model, tokenizer, texts, classifier, name, max_length=512, max_chunks=4):
    encoded_texts = tokenizer(
        texts,
        truncation=False,
        padding=True,
        return_tensors="pt"
    )

    input_ids = encoded_texts['input_ids'].cuda()
    attention_mask = encoded_texts['attention_mask'].cuda()

    batch_logits = []
    for i in range(len(input_ids)):
        input_id = input_ids[i]
        attention_mask_i = attention_mask[i]

        # Split the input into chunks of 512 tokens
        chunks = [input_id[j:j + 512] for j in range(0, len(input_id), 512)]
        attention_chunks = [attention_mask_i[j:j + 512] for j in range(0, len(input_id), 512)]

        if len(chunks) > max_chunks:
            chunks = chunks[:max_chunks]
            attention_chunks = attention_chunks[:max_chunks]

        last_chunk_size = len(chunks[-1])

        if last_chunk_size < 512:
            padding_length = 512 - last_chunk_size
            chunks[-1] = torch.cat([chunks[-1], torch.zeros(padding_length, dtype=torch.long).cuda()])
            attention_chunks[-1] = torch.cat([attention_chunks[-1], torch.zeros(padding_length, dtype=torch.long).cuda()])

        chunk_logits = []
        for chunk, mask in zip(chunks, attention_chunks):
            chunk = chunk.unsqueeze(0).cuda()
            mask = mask.unsqueeze(0).cuda()

            # Get model outputs
            outputs = model(input_ids=chunk, attention_mask=mask)
            cls_output = outputs.last_hidden_state[:, 0, :]

            logits = classifier(cls_output)
            chunk_logits.append(logits)

        chunk_logits = torch.cat(chunk_logits, dim=0)  # Shape: (num_chunks, 1)

        # Calculate weights based on chunk size
        chunk_weights = torch.ones(len(chunks), dtype=torch.float).cuda() * 512
        chunk_weights[-1] = last_chunk_size
        chunk_weights = chunk_weights / 512

        # Compute weighted logits
        weighted_logits = torch.sum(chunk_logits.squeeze() * chunk_weights) / torch.sum(chunk_weights)

        batch_logits.append(weighted_logits)

    final_logits = torch.stack(batch_logits)  # Shape: (batch_size,)
    return final_logits

# RoBERTa Model with classifier
class DistilSentinel(nn.Module):
    def __init__(self):
        super(DistilSentinel, self).__init__()
        self.distil = DistilBertModel.from_pretrained('distilbert-base-uncased')
        self.classifier = nn.Sequential(
            nn.Linear(768, 768),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(768, 1)
        )

    def forward(self, texts, tokenizer):
        logits = process_chunks(self.distil, tokenizer, texts, self.classifier, 'distilbert')
        return logits

# T5 Model with classifier
class T5BinaryClassifier(nn.Module):
    def __init__(self, model_name="t5-small"):
        super(T5BinaryClassifier, self).__init__()
        self.t5 = T5ForConditionalGeneration.from_pretrained(model_name)
        self.classifier = nn.Linear(self.t5.config.d_model, 1)

    def forward(self, texts, tokenizer):
        logits = process_chunks(self.t5.encoder, tokenizer, texts, self.classifier, 'T5')
        return logits

# Initialize models and tokenizers
distil_model = DistilSentinel().cuda()
distil_tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

t5_model = T5BinaryClassifier().cuda()
t5_tokenizer = T5Tokenizer.from_pretrained('t5-small')

# Ensemble model class
class EnsembleModel(nn.Module):
    def __init__(self, distil_model, t5_model, distil_tokenizer, t5_tokenizer):
        super(EnsembleModel, self).__init__()
        self.distil_model = distil_model
        self.t5_model = t5_model
        self.distil_tokenizer = distil_tokenizer
        self.t5_tokenizer = t5_tokenizer

    def forward(self, texts):
        # Get logits from both models
        distil_logits = self.distil_model(texts, self.distil_tokenizer)
        t5_logits = self.t5_model(texts, self.t5_tokenizer)

        # Compute final logits as average
        final_logits = torch.max(distil_logits, t5_logits)

        return distil_logits, t5_logits, final_logits

# Initialize the ensemble model
ensemble_model = EnsembleModel(distil_model, t5_model, distil_tokenizer, t5_tokenizer).cuda()

# Loss and optimizer
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(ensemble_model.parameters(), lr=1e-4)

# Training loop
num_epochs = 3
ensemble_model.train()

for epoch in range(num_epochs):
    running_loss = 0.0
    correct_predictions = 0
    total_predictions = 0

    for texts, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}"):
        texts = list(texts)  # Convert batch of texts to a list
        labels = labels.to("cuda").float()

        labels = labels.view(-1, 1)

        optimizer.zero_grad()

        # Forward pass through the ensemble model
        distil_logits, t5_logits, final_logits = ensemble_model(texts)
        final_logits = final_logits.view(-1, 1)
        # Compute the loss
        loss = criterion(final_logits, labels)

        # Backward pass and optimization
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        
        # Make predictions
        predicted = (final_logits > 0).float()
        correct_predictions += (predicted == labels).sum().item()
        total_predictions += labels.size(0)

        # Debugging logs
        for i in range(labels.size(0)):
            actual_class = labels[i].item()
            final_prob = torch.sigmoid(final_logits[i]).item()
            print(f"Data point {i+1}: Actual Class: {actual_class}, Final Logit: {final_logits[i].item()}, Predicted Probability: {final_prob:.4f}, Prediction: {predicted[i].item()}")

    epoch_loss = running_loss / len(train_loader)
    epoch_accuracy = correct_predictions / total_predictions
    print(f"Epoch {epoch+1}, Loss: {epoch_loss:.4f}, Accuracy: {epoch_accuracy:.4f}")
# Save the model at the end of each epoch
print("Saving the ensemble model")
torch.save(ensemble_model.state_dict(), f'our_ensemble_model.pth')
print(f"Model saved as 'ensemble_model_epoch.pth'")

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
Epoch 1/3:   0%|          | 1/4000 [00:00<24:17,  2.74it/s]Token indices sequence length is longer than the specified maximum sequence length for this model (3674 > 512). Running this sequence through the model will result in indexing errors


Data point 1: Actual Class: 0.0, Final Logit: -0.01138974353671074, Predicted Probability: 0.4972, Prediction: 0.0
Data point 2: Actual Class: 1.0, Final Logit: 0.016378028318285942, Predicted Probability: 0.5041, Prediction: 1.0


Token indices sequence length is longer than the specified maximum sequence length for this model (3960 > 512). Running this sequence through the model will result in indexing errors
Epoch 1/3:   0%|          | 2/4000 [00:01<42:47,  1.56it/s]

Data point 1: Actual Class: 0.0, Final Logit: -0.028333699330687523, Predicted Probability: 0.4929, Prediction: 0.0
Data point 2: Actual Class: 0.0, Final Logit: 0.03586166724562645, Predicted Probability: 0.5090, Prediction: 1.0


Epoch 1/3:   0%|          | 3/4000 [00:01<31:29,  2.12it/s]

...

Epoch 3/3: 100%|█████████▉| 3999/4000 [38:16<00:00,  1.55it/s]

Data point 1: Actual Class: 0.0, Final Logit: -10.090389251708984, Predicted Probability: 0.0000, Prediction: 0.0
Data point 2: Actual Class: 0.0, Final Logit: -11.176041603088379, Predicted Probability: 0.0000, Prediction: 0.0


Epoch 3/3: 100%|██████████| 4000/4000 [38:16<00:00,  1.74it/s]

Data point 1: Actual Class: 1.0, Final Logit: 7.1493940353393555, Predicted Probability: 0.9992, Prediction: 1.0
Data point 2: Actual Class: 0.0, Final Logit: -11.234585762023926, Predicted Probability: 0.0000, Prediction: 0.0
Epoch 3, Loss: 0.0478, Accuracy: 0.9865
Saving the ensemble model


Model saved as 'ensemble_model_epoch.pth'


In [37]:
from sklearn.metrics import classification_report, accuracy_score

# Evaluation function
def evaluate_model(model, test_loader, criterion):
    model.eval()  # Set the model to evaluation mode
    all_labels = []
    all_predictions = []
    total_loss = 0.0

    with torch.no_grad():
        for texts, labels in tqdm(test_loader, desc="Evaluating"):
            texts = list(texts)  # Convert batch of texts to a list
            labels = labels.to("cuda").float().view(-1, 1)

            # Forward pass through the ensemble model
            distil_logits, t5_logits, final_logits = model(texts)
            final_logits = final_logits.view(-1, 1)

            # Compute the loss
            loss = criterion(final_logits, labels)
            total_loss += loss.item()

            # Make predictions
            predictions = (final_logits > 0).float()

            # Store the true labels and predictions
            all_labels.extend(labels.cpu().numpy())
            all_predictions.extend(predictions.cpu().numpy())

    # Compute average loss over the test set
    avg_loss = total_loss / len(test_loader)

    # Compute accuracy
    accuracy = accuracy_score(all_labels, all_predictions)

    # Generate classification report
    report = classification_report(all_labels, all_predictions, target_names=["Class 0", "Class 1"])

    print(f"Test Loss: {avg_loss:.4f}")
    print(f"Test Accuracy: {accuracy:.4f}")
    print("Classification Report:")
    print(report)

    return avg_loss, accuracy, report

# Call the evaluation function
criterion = nn.BCEWithLogitsLoss()
test_loss, test_accuracy, classification_report = evaluate_model(ensemble_model, test_loader, criterion)

Evaluating: 100%|██████████| 1000/1000 [03:59<00:00,  4.17it/s]

Test Loss: 0.0935
Test Accuracy: 0.9740
Classification Report:
              precision    recall  f1-score   support

     Class 0       0.98      0.96      0.97       975
     Class 1       0.97      0.98      0.97      1025

    accuracy                           0.97      2000
   macro avg       0.97      0.97      0.97      2000
weighted avg       0.97      0.97      0.97      2000



# testing on other datasets

## base paper dataset(10k datapoints)

In [38]:
df_base_paper_dataset = pd.read_csv('/kaggle/input/humanllm/Final_dataset.csv')
df_base_paper_dataset = df_base_paper_dataset[:10000]
df_base_paper_dataset.drop(columns=['uid'], inplace=True)

In [39]:
df_base_paper_dataset.head()

,text,label
0,Italy's President Sergio Mattarella has dissol...,1
1,Brendan Gallagher's piano career was abruptly ...,1
2,The concept for Wetsleeve(tm) was born out of ...,0
3,Police officers in the United States had a tou...,1
4,A woman has filed a lawsuit seeking more than ...,1


In [40]:
class TextDataset(Dataset):
    def __init__(self, dataframe):
        self.dataframe = dataframe

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        text = self.dataframe.iloc[idx, 0]  # assuming 'text' is the first column
        label = int(self.dataframe.iloc[idx, 1])  # convert label to integer
        return text, label

# Data Loaders
# train_dataset = TextDataset(train_df)
test_dataset = TextDataset(df_base_paper_dataset)

# train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)  # Single datapoint per batch
test_loader = DataLoader(test_dataset, batch_size=2, shuffle=False)


In [41]:
import torch
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report

# Load the saved model weights
ensemble_model = EnsembleModel(distil_model, t5_model, distil_tokenizer, t5_tokenizer)
# ensemble_model.load_state_dict(torch.load('/kaggle/input/2_epoch_mix_model/other/default/1/our_ensemble_model.pth'))
# ensemble_model.cuda()
ensemble_model.eval()

# DataLoader for the test dataset
# test_loader = DataLoader(df_base_paper_dataset, batch_size=2, shuffle=False)

# Function to test the model on the test data
def test_model(model, dataloader):
    model.eval()
    predictions = []
    true_labels = []
    with torch.no_grad():
        for texts, labels in tqdm(dataloader, desc="Testing"):
            texts = list(texts)
            labels = labels.to("cuda").float()

            # Forward pass through the ensemble model
            _, _, final_logits = model(texts)
            final_probs = torch.sigmoid(final_logits).squeeze().cpu().numpy()

            # Convert probabilities to binary predictions (threshold at 0.5)
            predicted_classes = (final_probs >= 0.5).astype(int)

            # Store predictions and true labels
            predictions.extend(predicted_classes)
            true_labels.extend(labels.cpu().numpy())

    return predictions, true_labels

# Test the model and get predictions
predictions, true_labels = test_model(ensemble_model, test_loader)

# Generate the classification report
report = classification_report(true_labels, predictions, target_names=["Class 0", "Class 1"])
print(report)


Testing: 100%|██████████| 5000/5000 [11:27<00:00,  7.27it/s]

              precision    recall  f1-score   support

     Class 0       1.00      0.86      0.92      4952
     Class 1       0.88      1.00      0.93      5048

    accuracy                           0.93     10000
   macro avg       0.94      0.93      0.93     10000
weighted avg       0.94      0.93      0.93     10000



## HC3 dataset from hugging face(5k datapoints)

In [42]:
# !pip install datasets
from datasets import load_dataset

ds = load_dataset("Hello-SimpleAI/HC3", "all")

# Concatenate inner lists into a single string for each element
concatenated = [' '.join(text_list) for text_list in ds['train']['human_answers']]

# Create a DataFrame with a single column named 'text'
df_human_hc3 = pd.DataFrame(concatenated, columns=['text'])
df_human_hc3['label'] = 0
df_human_hc3 = df_human_hc3[:2500]


concatenated_ = [' '.join(text_list) for text_list in ds['train']['chatgpt_answers']]

# Create a DataFrame with a single column named 'text'
df_ai_hc3 = pd.DataFrame(concatenated_, columns=['text'])
df_ai_hc3['label'] = 1
df_ai_hc3 = df_ai_hc3[:2500]



df_HC3 = pd.concat([df_ai_hc3, df_human_hc3], ignore_index=True)
# df_HC3 = df_HC3[:100]

class TextDataset(Dataset):
    def __init__(self, dataframe):
        self.dataframe = dataframe

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        text = self.dataframe.iloc[idx, 0]  # assuming 'text' is the first column
        label = int(self.dataframe.iloc[idx, 1])  # convert label to integer
        return text, label

# Data Loaders
# train_dataset = TextDataset(train_df)
test_dataset = TextDataset(df_HC3)

# train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)  # Single datapoint per batch
test_loader = DataLoader(test_dataset, batch_size=2, shuffle=False)

In [ ]:
import torch
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report

# Load the saved model weights
ensemble_model = EnsembleModel(distil_model, t5_model, distil_tokenizer, t5_tokenizer)
# ensemble_model.load_state_dict(torch.load('/kaggle/input/2_epoch_mix_model/other/default/1/our_ensemble_model.pth'))
# ensemble_model.cuda()
ensemble_model.eval()

# DataLoader for the test dataset
# test_loader = DataLoader(df_HC3, batch_size=2, shuffle=False)

# Function to test the model on the test data
def test_model(model, dataloader):
    model.eval()
    predictions = []
    true_labels = []
    with torch.no_grad():
        for texts, labels in tqdm(dataloader, desc="Testing"):
            texts = list(texts)
            labels = labels.to("cuda").float()

            # Forward pass through the ensemble model
            _, _, final_logits = model(texts)
            final_probs = torch.sigmoid(final_logits).squeeze().cpu().numpy()

            # Convert probabilities to binary predictions (threshold at 0.5)
            predicted_classes = (final_probs >= 0.5).astype(int)

            # Store predictions and true labels
            predictions.extend(predicted_classes)
            true_labels.extend(labels.cpu().numpy())

    return predictions, true_labels

# Test the model and get predictions
predictions, true_labels = test_model(ensemble_model, test_loader)

# Generate the classification report
report = classification_report(true_labels, predictions, target_names=["Class 0", "Class 1"], labels=[0, 1])
print(report)


Testing:  78%|███████▊  | 1938/2500 [02:34<00:52, 10.77it/s]

# LIMA dataset

In [ ]:
from datasets import load_dataset

# Login using e.g. `huggingface-cli login` to access this dataset
ds = load_dataset("GAIR/lima")